# UNI-24 · BUSI: данные, EDA и воспроизводимый input pipeline

Этот ноутбук выполняет E0 из [плана экспериментов](../docs/baseline-experiment-plan.md): проверку данных, EDA, поиск дубликатов, split и подготовку Dataset/DataLoader. Обучения и оценки моделей здесь нет. Все результаты появятся **после вашего запуска**; ноутбук поставляется без outputs.

Источник: [Breast Ultrasound Images Dataset](https://www.kaggle.com/datasets/aryashah2k/breast-ultrasound-images-dataset).
Классы: **0 normal, 1 benign, 2 malignant**. Ожидаемые 133/437/210 — ориентир, а не подмена фактического аудита. Маски не являются входом классификатора.

Порядок: Run all → проверить invalid/conflicts и примеры → сохранить весь каталог artifacts на постоянное хранилище. В Colab локальный диск временный; перед повторным запуском восстановите сохранённые split CSV **и JSON**. Не создавайте новый split для каждого эксперимента.

## 1. Среда и настройки

Локально откройте ноутбук из checkout проекта; зависимости: `python -m pip install -r requirements-data.txt`. В Colab ячейка может клонировать ветку PR (после merge замените REPO_REF на master).

Установку можно включить флагом INSTALL_DEPENDENCIES. После установки/замены torch или numpy перезапустите kernel и начните заново с флагом False. GPU для подготовки данных не нужен. Не помещайте Kaggle token в notebook: используйте штатную авторизацию Kaggle или заранее распакованный локальный датасет.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess

REPO_REF = "codex/uni-24-data-pipeline"  # После merge можно использовать master.
INSTALL_DEPENDENCIES = False
IN_COLAB = "google.colab" in sys.modules

candidates = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next((p for p in candidates if (p / "busi/data.py").exists()), None)
if PROJECT_ROOT is None and IN_COLAB:
    PROJECT_ROOT = Path("/content/breast-cancer-classification")
    if not PROJECT_ROOT.exists():
        subprocess.run(["git", "clone", "--branch", REPO_REF,
                        "https://github.com/carbon77/breast-cancer-classification.git",
                        str(PROJECT_ROOT)], check=True)
if PROJECT_ROOT is None or not (PROJECT_ROOT / "busi/data.py").exists():
    raise RuntimeError("Откройте notebook из checkout проекта, содержащего busi/data.py")
PROJECT_ROOT = PROJECT_ROOT.resolve()
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r",
                    str(PROJECT_ROOT / "requirements-data.txt")], check=True)
sys.path.insert(0, str(PROJECT_ROOT))
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

# Локальная папка или None для загрузки выбранной версии через kagglehub.
LOCAL_DATA_ROOT = None  # Например Path("/content/drive/MyDrive/BUSI/Dataset_BUSI_with_GT")
KAGGLE_VERSION = 1
SPLIT_VERSION = "v1"
PATIENT_CSV = None  # Необязательно: CSV filepath,patient_id; полное покрытие оригиналов.
ARTIFACTS = PROJECT_ROOT / "artifacts"  # В Colab замените на путь в подключённом Drive.
RUN_DIR = ARTIFACTS / "data" / SPLIT_VERSION
RUN_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display
from busi.data import (
    CLASSES, CLASS_TO_IDX, DATASET_SLUG, SEED, IMAGE_SIZE, find_data_root,
    scan_dataset, group_images, freeze_split, train_statistics, class_weights,
    preprocessing_config, make_transform, make_loaders, seed_everything,
    environment, sha256, write_json, validate_split,
)

seed_everything(SEED)
if LOCAL_DATA_ROOT is None:
    import kagglehub
    dataset_path = kagglehub.dataset_download(f"{DATASET_SLUG}/versions/{KAGGLE_VERSION}")
    DATA_ROOT = find_data_root(dataset_path)
    source = {"dataset": DATASET_SLUG, "kaggle_version": KAGGLE_VERSION}
else:
    DATA_ROOT = find_data_root(LOCAL_DATA_ROOT)
    source = {"dataset": DATASET_SLUG, "kaggle_version": "local-user-supplied"}
git_commit = subprocess.run(["git", "-C", str(PROJECT_ROOT), "rev-parse", "HEAD"],
                            check=True, capture_output=True, text=True).stdout.strip()
write_json(RUN_DIR / "source.json", {**source, "code_commit": git_commit, "environment": environment()})
print("Dataset:", DATA_ROOT)
display(pd.Series(environment()))


## 2. Аудит структуры и качества

Все файлы попадут в inventory.csv с SHA-256, статусом декодирования, размерами и режимом каналов. Маски `_mask.png`, `_mask_1.png` и т. д. распознаются отдельно и проверяются, но исключаются из input pipeline. Повреждённые оригиналы исключаются с явным отчётом. Чужие файлы и вложенные папки отмечаются invalid.

Показатели яркости и контраста помогают выбрать кадры для просмотра; они не служат автоматическим медицинским фильтром. Отсутствие маски отмечается, но не мешает классификации.

In [ ]:
inventory = scan_dataset(DATA_ROOT)
inventory.to_csv(RUN_DIR / "inventory.csv", index=False)
invalid = inventory[inventory.status != "ok"]
invalid.to_csv(RUN_DIR / "invalid_files.csv", index=False)
originals = inventory[(inventory.status == "ok") & ~inventory.is_mask].copy()
summary = inventory.groupby(["class", "is_mask", "status"], dropna=False).size().rename("files").reset_index()
summary.to_csv(RUN_DIR / "audit_summary.csv", index=False)
display(summary)
display(invalid[["filepath", "error"]])
display(originals.groupby("class")[["width", "height", "channels", "mask_count"]].describe())
print("Оригиналы без масок:", int((originals.mask_count == 0).sum()))
print("Невалидных файлов:", len(invalid))


## 3. EDA: классы, каналы, размеры и примеры

Графики автоматически сохраняются в PNG для отчёта. Просмотрите подписи/рамки, встроенный текст, засветку и чёрные поля. Не меняйте pipeline по качеству на test: EDA ниже описывает исходный датасет до экспериментов, а подбор преобразований в дальнейшем выполняется только по train/validation.

In [ ]:
def save_figure(name):
    plt.tight_layout()
    plt.savefig(RUN_DIR / f"{name}.png", dpi=160, bbox_inches="tight")
    plt.show()
    plt.close()

counts = originals["class"].value_counts().reindex(CLASSES, fill_value=0)
class_report = pd.DataFrame({"count": counts, "fraction": counts / counts.sum()})
class_report.to_csv(RUN_DIR / "class_distribution.csv")
display(class_report)
print("Дисбаланс max/min:", round(counts.max() / counts.min(), 3))
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
counts.plot.bar(ax=axes[0], title="Классы (оригиналы)", rot=0)
axes[1].scatter(originals.width, originals.height, alpha=0.45)
axes[1].set(xlabel="Ширина, px", ylabel="Высота, px", title="Размеры")
originals["mode"].value_counts().plot.bar(ax=axes[2], title="Режим каналов", rot=0)
save_figure("eda_distribution")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for axis, field in zip(axes, ["mean_intensity", "std_intensity", "dark_fraction"]):
    for label in CLASSES:
        axis.hist(originals.loc[originals["class"] == label, field], bins=25, alpha=0.4, label=label)
    axis.set_title(field)
    axis.legend()
save_figure("eda_intensity")


In [ ]:
def gallery(rows, name, columns=3):
    rows = rows.reset_index(drop=True)
    if rows.empty:
        print(name, "— нет изображений")
        return
    nrows = (len(rows) + columns - 1) // columns
    fig, axes = plt.subplots(nrows, columns, figsize=(4 * columns, 3.5 * nrows), squeeze=False)
    for axis in axes.ravel():
        axis.axis("off")
    for axis, row in zip(axes.ravel(), rows.itertuples()):
        with Image.open(DATA_ROOT / row.filepath) as image:
            axis.imshow(image.convert("RGB"))
        axis.set_title(row.filepath, fontsize=8)
    save_figure(name)

examples = pd.concat([originals[originals["class"] == c].sample(min(3, int(counts[c])), random_state=SEED)
                      for c in CLASSES])
gallery(examples, "eda_examples")
suspects = pd.concat([originals.nsmallest(3, "std_intensity"),
                      originals.nlargest(3, "bright_fraction"),
                      originals.nlargest(3, "dark_fraction")]).drop_duplicates("filepath")
gallery(suspects, "eda_quality_review")


### Ручные наблюдения после запуска

Заполните после просмотра: какие артефакты найдены; какие файлы требуют исключения и почему; вывод о дисбалансе. Не удаляйте исходники молча. Если решения меняют состав уже зафиксированного split, нужна новая версия и повторное обучение всех моделей.

## 4. Дубликаты и группы

64-bit pHash: grayscale → 32×32 → DCT → 8×8 → порог медианы. Рёбра при расстоянии Хэмминга ≤4 или одинаковом SHA-256; связные компоненты сохраняют и транзитивные связи. Для малого BUSI используется прозрачный O(N²) поиск пар.

Если есть надёжные patient ID, передайте PATIENT_CSV: `filepath,patient_id`, путь относительно DATA_ROOT; CSV должен покрывать каждый валидный оригинал. Номер в имени файла **не** считается patient ID. Связи пациенток объединяются с дубликатами. Любая компонента с разными метками консервативно исключается целиком до ручного решения.

Без patient ID нельзя гарантировать отсутствие patient leakage: pHash контролирует лишь похожие снимки. Даже при группировке метрики могут быть завышены.

In [ ]:
grouped, duplicate_pairs = group_images(inventory, PATIENT_CSV, threshold=4)
grouped.to_csv(RUN_DIR / "grouped_inventory.csv", index=False)
duplicate_pairs.to_csv(RUN_DIR / "duplicate_pairs.csv", index=False)
conflicts = grouped[grouped.needs_review]
conflicts.to_csv(RUN_DIR / "conflicts.csv", index=False)
display(duplicate_pairs.head(20))
display(conflicts[["filepath", "class", "group_id"]])
print("Валидных оригиналов:", len(grouped), "групп:", grouped.group_id.nunique())
print("Точных пар:", int(duplicate_pairs.exact.sum()), "всех похожих пар:", len(duplicate_pairs))
print("Карантин:", len(conflicts), "изображений")
gallery(conflicts.head(12), "eda_conflicts")
if not duplicate_pairs.empty:
    review_paths = pd.unique(duplicate_pairs.head(6)[["filepath_a", "filepath_b"]].to_numpy().ravel())
    gallery(grouped[grouped.filepath.isin(review_paths)], "eda_duplicate_examples")


## 5. Стратифицированный group split 70/15/15, seed 42

Все компоненты однородны после карантина. Для каждого класса выбирается ближайшее по количеству изображений разделение из фиксированных кандидатов GroupShuffleSplit (128×32). Группы неделимы, поэтому доли приблизительны; при нехватке трёх независимых групп выполнение остановится.

split_v1.csv хранит `filepath,class,group_id,split`. JSON фиксирует SHA-256 split, fingerprint всех файлов (включая маски/битые), группировку, параметры, источник и версии библиотек. Повторный запуск загружает тот же split, сверяет содержимое и отказывается перезаписывать изменённые данные. Новые данные/решения → новая версия v2, а не ещё один случайный split.

In [ ]:
split, split_metadata = freeze_split(
    grouped, inventory, ARTIFACTS / "splits", version=SPLIT_VERSION, seed=SEED, threshold=4,
)
validate_split(split)
assert set(split.filepath) == set(grouped.loc[~grouped.needs_review, "filepath"])
assert split.groupby("group_id").split.nunique().max() == 1
assert not split.filepath.str.contains(r"_mask", case=False).any()
if "patient_id" in grouped:
    joined = split.merge(grouped[["filepath", "patient_id"]], on="filepath", validate="one_to_one")
    assert joined.groupby("patient_id").split.nunique().max() == 1
split_counts = pd.crosstab(split["class"], split.split).reindex(index=CLASSES, columns=["train", "validation", "test"])
split_counts.to_csv(RUN_DIR / "split_counts.csv")
display(split_counts)
display(split_counts.div(split_counts.sum(axis=1), axis=0).round(3))
split_counts.plot.bar(stacked=True, figsize=(8, 4), rot=0, title="Состав split")
save_figure("split_distribution")
print("Split SHA-256:", split_metadata["split_sha256"])
display(pd.Series(split_metadata))


## 6. Train-only statistics и preprocessing

224×224, RGB (grayscale дублируется по каналам), bilinear resize, tensor [0,1]. Mean/std baseline считаются исключительно по пикселям train после детерминированного resize. Для pretrained CNN/ViT доступна фиксированная ImageNet-нормализация: проверьте её соответствие конкретным выбранным весам в следующем эксперименте.

Только train: horizontal flip p=0.5, rotation ±10°, crop scale 0.85–1.0 без искажения aspect ratio, brightness/contrast ±10%. Вертикальный flip и hue/saturation отсутствуют. Validation/test всегда детерминированы. Config и split SHA сохраняйте в checkpoint; class weights вычисляются только по train.

In [ ]:
stats = train_statistics(DATA_ROOT, split)
config = preprocessing_config(stats, split_metadata["split_sha256"])
write_json(RUN_DIR / "train_statistics.json", stats)
write_json(RUN_DIR / "preprocessing.json", config)
weights = class_weights(split)
write_json(RUN_DIR / "class_weights.json", dict(zip(CLASSES, map(float, weights))))
display(pd.DataFrame(config["baseline"], index=["R", "G", "B"]))
display(pd.Series(dict(zip(CLASSES, weights)), name="train class weight"))

sample_path = split.loc[split.split == "train", "filepath"].iloc[0]
with Image.open(DATA_ROOT / sample_path) as image:
    sample = image.convert("RGB")
transform = make_transform(config, "train")
mean = np.array(config["baseline"]["mean"])
std = np.array(config["baseline"]["std"])
seed_everything(SEED)
fig, axes = plt.subplots(1, 5, figsize=(16, 4))
axes[0].imshow(sample)
axes[0].set_title("Оригинал train")
for axis in axes[1:]:
    tensor = transform(sample)
    axis.imshow(np.clip(tensor.permute(1, 2, 0).numpy() * std + mean, 0, 1))
    axis.set_title("Train augmentation")
for axis in axes:
    axis.axis("off")
save_figure("train_augmentations")


## 7. Dataset/DataLoader и проверки контракта

Ниже выполняется только smoke check input pipeline, без обучения и подсчёта model metrics. num_workers=0 подходит для Windows/Colab; worker seeds поддерживаются при увеличении числа workers. Для повторения batches создавайте новые loaders с тем же seed и вызывайте seed_everything перед итерацией. Сохранённый split не зависит от seed обучения.

In [ ]:
import torch

seed_everything(SEED)
loaders = make_loaders(DATA_ROOT, split, config, normalization="baseline", seed=SEED, num_workers=0)
images, labels = next(iter(loaders["train"]))
assert images.ndim == 4 and tuple(images.shape[1:]) == (3, IMAGE_SIZE, IMAGE_SIZE)
assert torch.isfinite(images).all() and set(labels.tolist()) <= {0, 1, 2}
for name in ("validation", "test"):
    dataset = loaders[name].dataset
    assert torch.equal(dataset[0][0], dataset[0][0])
print("Batch:", tuple(images.shape), "labels:", labels.tolist())
print("Input pipeline smoke checks: OK")
print("Loss weights:", torch.tensor(weights))
# Для ResNet/ViT:
# loaders = make_loaders(DATA_ROOT, split, config, normalization="imagenet", seed=43)


## 8. Версионирование и передача в обучение

Эквивалент лёгкого content-addressed manifest: inventory содержит SHA-256 каждого raw-файла; split JSON связывает dataset fingerprint, группировку и split; итоговый manifest связывает отчёты и preprocessing с code commit и окружением. Данные, графики, веса и токены игнорируются Git. Манифест проверяет целостность, но не заменяет хранилище: сохраните raw dataset выбранной версии Kaggle и **весь** artifacts во внешнем постоянном хранилище. DVC/remote не настроены.

Для ревью данных можно отдельно коммитить разрешённые текстовые манифесты; patient_id не публикуйте. Для model checkpoint в дальнейшем сохраняйте SHA-256 весов и preprocessing/split metadata рядом с файлом. Test остаётся для финальной оценки; здесь не получены никакие метрики моделей.

In [ ]:
manifest = {
    "schema_version": 1, "source": source, "code_commit": git_commit,
    "split_version": SPLIT_VERSION, "split_sha256": split_metadata["split_sha256"],
    "dataset_sha256": split_metadata["dataset_sha256"],
    "environment": environment(),
    "files": {
        p.relative_to(ARTIFACTS).as_posix(): sha256(p)
        for p in sorted(RUN_DIR.rglob("*"))
        if p.is_file() and p.name != "manifest.json"
    },
}
for suffix in ("csv", "json"):
    p = ARTIFACTS / "splits" / f"split_{SPLIT_VERSION}.{suffix}"
    manifest["files"][p.relative_to(ARTIFACTS).as_posix()] = sha256(p)
write_json(RUN_DIR / "manifest.json", manifest)
assert all(sha256(ARTIFACTS / name) == digest for name, digest in manifest["files"].items())
print("Готово:", RUN_DIR)
print("Сохраните raw dataset и artifacts на постоянный диск перед завершением Colab.")


### Что проверить вручную

- Проверить invalid_files.csv, конфликты меток и gallery похожих снимков.
- Описать артефакты качества и дисбаланс в разделе ручных наблюдений.
- Проверить фактические доли split и отсутствие разнесённых patient/group ID.
- Убедиться, что умеренные аугментации не нарушают структуру снимков.
- Сохранить artifacts и результаты своего запуска; повторить запуск с теми же данными.
- Отдельно задокументировать принятые решения по карантину перед началом E1.

Ограничения: нет клинической валидации; pHash даёт false positives/negatives, качество групп требует ручного просмотра. Доли могут отличаться от целевых из-за размеров групп. Воспроизводимость чисел между платформами требует совпадающих версий зависимостей.